# Malware Behavior Classification Model

## Purpose
This notebook builds a **multi-class classifier** that predicts a malware sample's
**behavior tag** (e.g. `backdoor`, `worm`, `downloader`, `ransom`) using **LightGBM**
(`LGBMClassifier`), trained on the same engineered static features as the detection model.

## Data
Labels and features live in **separate files**, joined on `sha256`:
- `win32_behavior_train_20pct.parquet` / `win32_test_behavior.parquet` — behavior tags (labels)
- `win32_detection_train_20pct.parquet` / `win32_test_detection.parquet` — `general`, `strings`, `imports` (features)

## Target preparation
The `behavior` column is a multi-label list assigned by ClarAVy:
1. **Untagged samples are dropped** — ~51% of rows carry only an empty-string
   placeholder meaning "no confident tag" (not "benign"). It isn't a real behavioral
   category, so it can't serve as a target class.
2. **Multi-tag samples are kept** — rows with 2+ tags (~10k) use their **first listed
   tag** as the primary label.
3. **No minimum class count** — every tag that appears in training becomes a class
   (~70 classes). The only exclusion is on the *test* side: samples whose tag never
   appears in training can't be scored and are removed from evaluation.

## Known limitation: class imbalance
`backdoor` alone is ~37% of the training target, while the rarest classes have only a
handful of samples. Expect strong performance on frequent classes (backdoor, packed,
downloader) and near-zero recall on rare ones — the classification report's **macro**
averages (treating every class equally) will sit far below the **weighted** ones.
Ideas to improve: `class_weight="balanced"`, richer features (full import vectors),
or collapsing rare tags into an "other" class.

In [1]:
# imports
from pathlib import Path
import json
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

### Load behavior labels (train + test)

In [2]:
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "win32_data"

beh_train_df = pd.read_parquet(DATA_DIR / "win32_behavior_train_20pct.parquet")
beh_test_df = pd.read_parquet(DATA_DIR / "win32_test_behavior.parquet")

print("behavior train rows:", len(beh_train_df))
print("behavior test rows:", len(beh_test_df))
display(beh_train_df.head())

behavior train rows: 156357
behavior test rows: 180000


,sha256,label,family,behavior,mbc,ttps
0,0015cbb011bbeb891b3ea505a6995d17395c4d4fd144b0...,1,berbew,[backdoor],[],[]
1,0020532d7497a685cf1d75668eef3184a0e683fd620090...,1,cosmu,[ransom],[],[]
2,00211af4c69b533a0663801dbb36617db6cc895ae3c6c8...,1,ctsinf,[],[],[]
3,0031339853d8f74ab0b22d4ea1501a43b8b2e28d972357...,1,koceg,[backdoor],[],[]
4,00328cb771644f2b8f05f76c4dda84179db343705d5141...,1,black,[worm],[],[]


### Build the target: one behavior class per sample

- No real tag → dropped (ClarAVy's empty-string placeholder means "no confident
  tag", not a behavior)
- One or more tags → the **first listed tag** (primary behavior)

In [3]:
def primary_tag(tags):
    """First real tag if the sample has any, else None (dropped)."""
    real_tags = [t for t in tags if t]
    return real_tags[0] if real_tags else None


beh_train_df["behavior_class"] = beh_train_df["behavior"].apply(primary_tag)
beh_test_df["behavior_class"] = beh_test_df["behavior"].apply(primary_tag)

train_tagged = beh_train_df.dropna(subset=["behavior_class"])
test_tagged = beh_test_df.dropna(subset=["behavior_class"])

print(f"train samples with a tag: {len(train_tagged)} / {len(beh_train_df)}")
print(f"test samples with a tag: {len(test_tagged)} / {len(beh_test_df)}")
print("\ntrain class distribution (top 20):")
display(train_tagged["behavior_class"].value_counts().head(20))

train samples with a tag: 76612 / 156357
test samples with a tag: 99531 / 180000

train class distribution (top 20):


behavior_class
backdoor           27766
worm               12792
virus              10191
downloader          7660
dropper             2786
adware              2044
selfmod             1972
ransom              1935
spyware             1728
packed              1575
injector             931
passwordstealer      711
stealer              705
pua                  604
coinminer            345
hacktool             293
servstart            216
cryptor              197
ddos                 176
gamehack             158
Name: count, dtype: int64

In [4]:
# every tag seen in training becomes a class; test samples whose tag never
# appears in training can't be scored, so they're excluded from evaluation only
train_classes = set(train_tagged["behavior_class"].unique())
test_in_train = test_tagged["behavior_class"].isin(train_classes)

train_labels = train_tagged[["sha256", "behavior_class"]].copy()
test_labels = test_tagged.loc[test_in_train, ["sha256", "behavior_class"]].copy()

print("classes:", len(train_classes))
print("train samples:", len(train_labels))
print("test samples:", len(test_labels),
      f"({(~test_in_train).sum()} excluded — class unseen in training)")

classes: 77
train samples: 76612
test samples: 99513 (18 excluded — class unseen in training)


In [5]:
# encode class names as integers for LightGBM
label_encoder = LabelEncoder()
label_encoder.fit(sorted(train_classes))

train_labels["y"] = label_encoder.transform(train_labels["behavior_class"])
test_labels["y"] = label_encoder.transform(test_labels["behavior_class"])

### Join features from the detection files

The behavior files have no feature columns — `general`, `strings`, and `imports`
come from the detection parquets, joined on `sha256`. (Every behavior-train sha256
appears in the detection train file.)

In [6]:
det_train_df = pd.read_parquet(
    DATA_DIR / "win32_detection_train_20pct.parquet",
    columns=["sha256", "general", "strings", "imports"]
)
det_test_df = pd.read_parquet(
    DATA_DIR / "win32_test_detection.parquet",
    columns=["sha256", "general", "strings", "imports"]
)

train_merged = train_labels.merge(det_train_df, on="sha256", how="inner")
test_merged = test_labels.merge(det_test_df, on="sha256", how="inner")

print("train merged:", train_merged.shape)
print("test merged:", test_merged.shape)

train merged: (76612, 6)
test merged: (99513, 6)


### Feature Engineering Functions

Same features as `detection-model.ipynb`: general file metadata, string statistics,
and import summaries.

In [7]:
def extract_import_features(x):
    """
    Convert imports JSON into compact numerical features.
    """
    if isinstance(x, str):
        x = json.loads(x)

    if not isinstance(x, dict):
        return {}

    features = {}

    # number of imported DLLs
    features["num_imported_dlls"] = len(x)

    # total imported functions
    features["num_imported_functions"] = sum(
        len(funcs) for funcs in x.values()
    )

    # Average functions per DLL
    if len(x) > 0:
        features["avg_functions_per_dll"] = (
            features["num_imported_functions"] / len(x)
        )
    else:
        features["avg_functions_per_dll"] = 0

    # DLL presence indicators
    common_dlls = [
        "KERNEL32.dll",
        "ADVAPI32.dll",
        "USER32.dll",
        "WS2_32.dll",
        "SHELL32.dll",
        "OLE32.dll"
    ]

    for dll in common_dlls:
        features[f"has_{dll}"] = int(dll in x)

    return features

In [8]:
def extract_string_features(x):
    if isinstance(x, str):
        x = json.loads(x)

    return {
        "numstrings": x.get("numstrings", 0),
        "avg_string_length": x.get("avlength", 0),
        "num_printables": x.get("printables", 0),
        "string_entropy": x.get("entropy", 0),
    }

In [9]:
def extract_general_features(x):
    if isinstance(x, str):
        x = json.loads(x)

    return {
        "file_size": x.get("size", 0),
        "file_entropy": x.get("entropy", 0),
        "is_pe": x.get("is_pe", 0),
    }

### Build X_train, y_train, X_test, and y_test

In [10]:
def build_features(df):
    imports_features = pd.DataFrame(df["imports"].apply(extract_import_features).tolist())
    strings_features = pd.DataFrame(df["strings"].apply(extract_string_features).tolist())
    general_features = pd.DataFrame(df["general"].apply(extract_general_features).tolist())

    return pd.concat(
        [
            general_features,
            strings_features,
            imports_features
        ],
        axis=1
    )


X_train = build_features(train_merged)
y_train = train_merged["y"]

X_test = build_features(test_merged)
y_test = test_merged["y"]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (76612, 16)
X_test shape: (99513, 16)


### Training/Testing Model

In [11]:
model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    verbosity=-1
)

model.fit(X_train, y_train)

,random_state,42
,verbosity,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [12]:
# predict behavior classes for the test set
y_pred = model.predict(X_test)

### Evaluation

Accuracy plus a per-class report. With ~70 imbalanced classes, the **macro** averages
(treating every class equally) will sit far below the **weighted** ones — that gap
measures how much the model leans on the frequent classes.

In [13]:
print("Accuracy:", accuracy_score(y_test, y_pred))

# restrict the report to classes that actually appear in y_test or y_pred
present_classes = sorted(set(y_test) | set(y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=present_classes,
    target_names=[label_encoder.classes_[i] for i in present_classes],
    zero_division=0
))

Accuracy: 0.5686895179524283

Classification Report:
                 precision    recall  f1-score   support

         adware       0.08      0.00      0.00      2825
         antiav       0.00      0.00      0.00        23
         antifw       0.00      0.00      0.00        14
         antivm       0.00      0.00      0.00         8
        autorun       0.00      0.00      0.00       266
         avkill       0.00      0.00      0.00         1
       backdoor       0.73      0.94      0.82     46552
         banker       0.00      0.00      0.00       348
            bho       0.00      0.00      0.00        88
         binder       0.00      0.00      0.00        24
        blocker       0.00      0.00      0.00       175
      bypassuac       0.00      0.00      0.00         9
        clicker       0.00      0.00      0.00       210
      coinminer       0.00      0.00      0.00       852
      cracktool       0.00      0.00      0.00       191
        cryptor       0.00      0.

In [14]:
# per-class test accuracy at a glance: rows = true class, columns = predicted
cm = confusion_matrix(y_test, y_pred, labels=range(len(label_encoder.classes_)))
cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)

# show the 10 most frequent true classes only, to keep the matrix readable
top_true = cm_df.sum(axis=1).sort_values(ascending=False).head(10).index
display(cm_df.loc[top_true, top_true])

,backdoor,virus,worm,downloader,spyware,adware,dropper,packed,injector,stealer
backdoor,43884,2199,33,109,1,0,13,191,1,9
virus,4274,8499,446,347,2,15,261,349,17,114
worm,2335,1977,1314,1141,3,4,49,634,8,21
downloader,1965,1745,122,2789,0,0,30,148,1,9
spyware,2676,947,29,200,0,1,10,43,0,1
adware,780,1937,11,11,1,6,7,7,1,5
dropper,907,930,56,125,0,20,42,35,1,22
packed,48,1522,2,23,0,0,1,19,0,13
injector,347,833,20,141,0,2,9,28,0,47
stealer,236,898,24,9,1,4,7,1,0,5
